<a href="https://colab.research.google.com/github/mschlemper/AtividadesFaculdade/blob/main/_supremo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Instalação de Dependências e Configuração do Ambiente

In [37]:
!pip install fastapi uvicorn pydantic scikit-learn joblib httpx nest_asyncio

### Fase 1: Preparação, Treinamento e Serialização do Modelo (`Pipeline Offline`)

Aqui, criamos um pipeline completo do scikit-learn para classificação de risco de crédito (Score), simulando dados históricos, dividindo em treino/teste, aplicando pré-processadores e salvando o artefato `.joblib` em um diretório dedicado.

In [38]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

# 1. Criação de Dados Sintéticos para o cenário de Risco de Crédito
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'idade': np.random.randint(18, 70, size=n_samples),
    'renda_anual': np.random.randint(15000, 150000, size=n_samples),
    'historico_inadimplencia': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]),
    'score_credito': np.random.randint(300, 850, size=n_samples)
})

# Definir variável alvo simples
data['inadimplente'] = ((data['historico_inadimplencia'] == 1) |
                        (data['score_credito'] < 500) |
                        (data['renda_anual'] < 25000)).astype(int)

X = data.drop(columns=['inadimplente'])
y = data['inadimplente']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Construção do Pipeline Robusto do Scikit-Learn
numeric_features = ['idade', 'renda_anual', 'score_credito']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('passthrough', 'passthrough', ['historico_inadimplencia'])
    ]
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Treinamento
model_pipeline.fit(X_train, y_train)

# Avaliação das Métricas (Fase 1 - Relatório de Métricas)
y_pred = model_pipeline.predict(X_test)
print("=== RELATÓRIO DE MÉTRICAS DO MODELO ===")
print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

# Criando diretório de persistência
os.makedirs('models', exist_ok=True)
model_path = 'models/credit_model.joblib'
joblib.dump(model_pipeline, model_path)
print(f"Modelo salvo com sucesso em: '{model_path}'")

=== RELATÓRIO DE MÉTRICAS DO MODELO ===
Acurácia: 1.0000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        93
           1       1.00      1.00      1.00       107

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Modelo salvo com sucesso em: 'models/credit_model.joblib'


### Fases 2, 3 e 4: Modelagem Pydantic, Arquitetura da API FastAPI e Ciclo de Vida (`Lifespan`)

Para reproduzir fielmente a estrutura modular exigida na avaliação diretamente neste Jupyter Notebook, organizaremos os blocos de código correspondentes a cada módulo do sistema.

In [39]:
from fastapi import FastAPI, APIRouter, HTTPException, Depends, status
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager
import joblib

# =====================================================================
# FASE 4: SCHEMAS (schemas.py ou schemas/)
# Validação de Tipagem, limites (Field) e exemplos documentados.
# =====================================================================

class PredictInput(BaseModel):
    idade: int = Field(..., ge=18, le=120, description="Idade do solicitante de crédito em anos.", json_schema_extra={"example": 35})
    renda_anual: float = Field(..., gt=0, description="Renda bruta anual do solicitante.", json_schema_extra={"example": 55000.0})
    historico_inadimplencia: int = Field(..., ge=0, le=1, description="Possui histórico de inadimplência? (1 para sim, 0 para não)", json_schema_extra={"example": 0})
    score_credito: int = Field(..., ge=300, le=850, description="Score de crédito do solicitante (Serasa/SPC)", json_schema_extra={"example": 720})

class PredictOutput(BaseModel):
    inadimplente_predito: int = Field(..., description="Predição final do modelo (1 para inadimplente, 0 para bom pagador).")
    probabilidade: float = Field(..., description="Probabilidade estimada do evento de inadimplência.")

# =====================================================================
# FASE 2 & 3: SERVICES & LIFESPAN (services.py & main.py)
# Modelo carregado apenas UMA VEZ na memória no ciclo startup/lifespan.
# =====================================================================

# Container global estável para armazenar referências em memória RAM
ml_models = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Inicialização (Fase 3 - Carregamento único em memória)
    model_file = 'models/credit_model.joblib'
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"O artefato do modelo não foi localizado em {model_file}")

    print("[LIFESPAN] Carregando o pipeline do modelo de Machine Learning para a memória RAM...")
    ml_models["credit_pipeline"] = joblib.load(model_file)
    print("[LIFESPAN] Modelo pronto para inferência.")

    yield

    # Encerramento do ciclo de vida da API
    print("[LIFESPAN] Limpando memória cache operacional...")
    ml_models.clear()

# Instanciação da aplicação FastAPI principal
app = FastAPI(
    title="API de Análise Preditiva de Risco de Crédito",
    description="Microsserviço de Machine Learning para predição de risco financeiro.",
    version="1.0.0",
    lifespan=lifespan
)

### Fase 5: Implementação de Endpoints (`APIRouter` e Tratamento de Erros)

In [40]:
router = APIRouter()

# 1. Endpoint GET /health
@router.get("/health", tags=["Monitoramento"])
async def health_check():
    if "credit_pipeline" not in ml_models:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="Modelo preditivo não carregado na memória ou corrompido."
        )
    return {
        "status": "healthy",
        "model_loaded": True,
        "model_path": "models/credit_model.joblib"
    }

# 2. Endpoint POST /predict (Fase 5 - Inferência e Tratamento de Exceções)
@router.post("/predict", response_model=PredictOutput, tags=["Inferência"])
async def predict(payload: PredictInput):
    try:
        # Recuperar o modelo instanciado pelo Lifespan sem tocar no HD a cada requisição!
        pipeline = ml_models.get("credit_pipeline")
        if not pipeline:
            raise HTTPException(
                status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
                detail="O modelo não está instanciado na memória."
            )

        # Converter o payload validado em DataFrame estruturado para as colunas do Scikit-Learn
        input_data = pd.DataFrame([payload.model_dump()])

        # Executar inferências preditivas
        prediction = int(pipeline.predict(input_data)[0])
        probabilities = pipeline.predict_proba(input_data)[0]
        confidence = float(probabilities[prediction])

        return PredictOutput(
            inadimplente_predito=prediction,
            probabilidade=confidence
        )

    except Exception as e:
        # Proteção robusta contra crash da API
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Ocorreu uma inconsistência no processamento dos dados estruturados: {str(e)}"
        )

# Inserindo o roteador na aplicação principal
app.include_router(router)

### Execução e Teste da API em Tempo Real (Direto no Colab)

Vamos habilitar o loop assíncrono interno do Colab usando `nest_asyncio` e subir a API FastAPI em background usando Threads para podermos testar os endpoints com dados simulados reais.

In [41]:
import threading
import uvicorn
import nest_asyncio
import httpx
import time

nest_asyncio.apply()

# Porta local de testes para o Colab
PORT = 8000

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="info")

# Iniciando servidor assíncrono em Thread separada
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Aguardar inicialização do lifespan
time.sleep(2.0)

INFO:     Started server process [868]
INFO:     Waiting for application startup.


[LIFESPAN] Carregando o pipeline do modelo de Machine Learning para a memória RAM...


INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


[LIFESPAN] Modelo pronto para inferência.
[LIFESPAN] Limpando memória cache operacional...


### Simulação de Requisições via HTTPX

Vamos agora verificar se o FastAPI valida o contrato Pydantic (bloqueando dados inválidos com `422 Unprocessable Entity`) e processa corretamente requisições válidas.

### 1. Criando a estrutura de diretórios do módulo da API

In [42]:
# Criando o pacote python 'app' para organização modular
!mkdir -p app
!touch app/__init__.py
print("Diretório app/ inicializado como pacote Python.")

Diretório app/ inicializado como pacote Python.


### 2. Criando o arquivo `app/schemas.py`

Contém a definição estrita do contrato de dados recebidos e enviados pela API com validações do `Pydantic`.

In [43]:
%%writefile app/schemas.py
from pydantic import BaseModel, Field

class PredictInput(BaseModel):
    idade: int = Field(..., ge=18, le=120, description="Idade do solicitante de crédito em anos.", json_schema_extra={"example": 35})
    renda_anual: float = Field(..., gt=0, description="Renda bruta anual do solicitante.", json_schema_extra={"example": 55000.0})
    historico_inadimplencia: int = Field(..., ge=0, le=1, description="Possui histórico de inadimplência? (1 para sim, 0 para não)", json_schema_extra={"example": 0})
    score_credito: int = Field(..., ge=300, le=850, description="Score de crédito do solicitante (Serasa/SPC)", json_schema_extra={"example": 720})

class PredictOutput(BaseModel):
    inadimplente_predito: int = Field(..., description="Predição final do modelo (1 para inadimplente, 0 para bom pagador).")
    probabilidade: float = Field(..., description="Probabilidade estimada do evento de inadimplência.")


Overwriting app/schemas.py


### 3. Criando o arquivo `app/services.py`

Isola o gerenciamento do estado em memória RAM e a manipulação direta do modelo carregado.

In [44]:
%%writefile app/services.py
import os
import joblib

# Container que guardará o modelo na memória RAM
ml_models = {}

def carregar_modelo():
    model_file = 'models/credit_model.joblib'
    if not os.path.exists(model_file):
        raise FileNotFoundError(f"O artefato do modelo não foi localizado em {model_file}")

    print("[LIFESPAN] Carregando o pipeline do modelo de Machine Learning para a memória RAM...")
    ml_models["credit_pipeline"] = joblib.load(model_file)
    print("[LIFESPAN] Modelo pronto para inferência.")

def limpar_modelo():
    print("[LIFESPAN] Limpando memória cache operacional...")
    ml_models.clear()

def obter_pipeline():
    return ml_models.get("credit_pipeline")


Overwriting app/services.py


### 4. Criando o arquivo `app/routers.py`

Isola as rotas usando o `APIRouter` do FastAPI.

In [45]:
%%writefile app/routers.py
import pandas as pd
from fastapi import APIRouter, HTTPException, status
from app.schemas import PredictInput, PredictOutput
from app.services import obter_pipeline

router = APIRouter()

@router.get("/health", tags=["Monitoramento"])
async def health_check():
    pipeline = obter_pipeline()
    if not pipeline:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="Modelo preditivo não carregado na memória ou corrompido."
        )
    return {
        "status": "healthy",
        "model_loaded": True,
        "model_path": "models/credit_model.joblib"
    }

@router.post("/predict", response_model=PredictOutput, tags=["Inferência"])
async def predict(payload: PredictInput):
    try:
        pipeline = obter_pipeline()
        if not pipeline:
            raise HTTPException(
                status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
                detail="O modelo não está instanciado na memória."
            )

        # Converter o payload validado em DataFrame estruturado
        input_data = pd.DataFrame([payload.model_dump()])

        # Executar inferências
        prediction = int(pipeline.predict(input_data)[0])
        probabilities = pipeline.predict_proba(input_data)[0]
        confidence = float(probabilities[prediction])

        return PredictOutput(
            inadimplente_predito=prediction,
            probabilidade=confidence
        )

    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Ocorreu uma inconsistência no processamento: {str(e)}"
        )


Overwriting app/routers.py


### 5. Criando o arquivo principal `app/main.py`

Configura e inicializa a aplicação conectando o ciclo de vida (`lifespan`) e os roteadores de endpoints.

In [46]:
%%writefile app/main.py
from fastapi import FastAPI
from contextlib import asynccontextmanager
from app.services import carregar_modelo, limpar_modelo
from app.routers import router

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Inicialização da aplicação (Fase 3)
    carregar_modelo()
    yield
    # Encerramento da aplicação
    limpar_modelo()

app = FastAPI(
    title="API de Análise Preditiva de Risco de Crédito",
    description="Microsserviço de Machine Learning para predição de risco financeiro.",
    version="1.0.0",
    lifespan=lifespan
)

# Inclui as rotas expostas pelo roteador
app.include_router(router)


Overwriting app/main.py


### 6. Gerando o arquivo `requirements.txt`

Fundamental para garantir a reprodutibilidade descrita na Seção 5.1 do seu projeto.

In [47]:
%%writefile requirements.txt
fastapi>=0.110.0
uvicorn>=0.28.0
pydantic>=2.6.0
scikit-learn>=1.4.0
joblib>=1.3.0
pandas>=2.0.0
numpy>=1.24.0


Overwriting requirements.txt


In [48]:
import json

print("=== TESTANDO ENDPOINT: /health ===")
response_health = httpx.get(f"http://127.0.0.1:{PORT}/health")
print(f"Status Code: {response_health.status_code}")
print(response_health.json())

print("\n=== TESTANDO ENDPOINT: /predict (PAYLOAD VÁLIDO) ===")
payload_valido = {
    "idade": 38,
    "renda_anual": 75000.0,
    "historico_inadimplencia": 0,
    "score_credito": 710
}

response_predict = httpx.post(f"http://127.0.0.1:{PORT}/predict", json=payload_valido)
print(f"Status Code: {response_predict.status_code}")
print(response_predict.json())

print("\n=== TESTANDO ENDPOINT: /predict (VALOR DA IDADE INVÁLIDO < 18 - ERRO PYDANTIC) ===")
payload_invalido = {
    "idade": 12,  # Regra de negócio mínima no schema: ge=18
    "renda_anual": 25000.0,
    "historico_inadimplencia": 0,
    "score_credito": 600
}

response_err = httpx.post(f"http://127.0.0.1:{PORT}/predict", json=payload_invalido)
print(f"Status Code: {response_err.status_code}")
print(json.dumps(response_err.json(), indent=2))

=== TESTANDO ENDPOINT: /health ===
INFO:     127.0.0.1:59610 - "GET /health HTTP/1.1" 503 Service Unavailable
Status Code: 503
{'detail': 'Modelo preditivo não carregado na memória ou corrompido.'}

=== TESTANDO ENDPOINT: /predict (PAYLOAD VÁLIDO) ===
INFO:     127.0.0.1:59620 - "POST /predict HTTP/1.1" 400 Bad Request
Status Code: 400
{'detail': 'Ocorreu uma inconsistência no processamento dos dados estruturados: 500: O modelo não está instanciado na memória.'}

=== TESTANDO ENDPOINT: /predict (VALOR DA IDADE INVÁLIDO < 18 - ERRO PYDANTIC) ===
INFO:     127.0.0.1:59634 - "POST /predict HTTP/1.1" 422 Unprocessable Content
Status Code: 422
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "body",
        "idade"
      ],
      "msg": "Input should be greater than or equal to 18",
      "input": 12,
      "ctx": {
        "ge": 18
      }
    }
  ]
}


### Servidor de Teste Alternativo (Porta 8001)

Para evitar o conflito de portas, vamos rodar a API de testes na porta `8001`.

In [50]:
import threading
import uvicorn
import nest_asyncio
import httpx
import time

nest_asyncio.apply()

# Usando a porta 8001 para evitar conflitos
PORT_ALT = 8001

def run_alt_server():
    # uvicorn carrega a app e ativa o lifespan automáticamente
    uvicorn.run(app, host="127.0.0.1", port=PORT_ALT, log_level="info")

# Iniciando servidor
alt_server_thread = threading.Thread(target=run_alt_server, daemon=True)
alt_server_thread.start()

# Aguardar inicialização e carga do lifespan em memória RAM
time.sleep(3.0)

INFO:     Started server process [868]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


[LIFESPAN] Carregando o pipeline do modelo de Machine Learning para a memória RAM...
[LIFESPAN] Modelo pronto para inferência.


### Testando novamente os endpoints na Porta 8001

In [51]:
import json

print("=== RE-TESTANDO ENDPOINT: /health ===")
res_health = httpx.get(f"http://127.0.0.1:{PORT_ALT}/health")
print(f"Status Code: {res_health.status_code}")
print(res_health.json())

print("\n=== RE-TESTANDO ENDPOINT: /predict (PAYLOAD VÁLIDO) ===")
payload_ok = {
    "idade": 35,
    "renda_anual": 55000.0,
    "historico_inadimplencia": 0,
    "score_credito": 720
}
res_predict = httpx.post(f"http://127.0.0.1:{PORT_ALT}/predict", json=payload_ok)
print(f"Status Code: {res_predict.status_code}")
print(res_predict.json())

print("\n=== RE-TESTANDO ENDPOINT: /predict (VALOR DA IDADE INVÁLIDO - ERRO 422) ===")
payload_nok = {
    "idade": 15,  # Inválido (menor que 18)
    "renda_anual": 30000.0,
    "historico_inadimplencia": 0,
    "score_credito": 620
}
res_predict_err = httpx.post(f"http://127.0.0.1:{PORT_ALT}/predict", json=payload_nok)
print(f"Status Code: {res_predict_err.status_code}")
print(json.dumps(res_predict_err.json(), indent=2))

=== RE-TESTANDO ENDPOINT: /health ===
INFO:     127.0.0.1:48648 - "GET /health HTTP/1.1" 200 OK
Status Code: 200
{'status': 'healthy', 'model_loaded': True, 'model_path': 'models/credit_model.joblib'}

=== RE-TESTANDO ENDPOINT: /predict (PAYLOAD VÁLIDO) ===
INFO:     127.0.0.1:48658 - "POST /predict HTTP/1.1" 200 OK
Status Code: 200
{'inadimplente_predito': 0, 'probabilidade': 1.0}

=== RE-TESTANDO ENDPOINT: /predict (VALOR DA IDADE INVÁLIDO - ERRO 422) ===
INFO:     127.0.0.1:48664 - "POST /predict HTTP/1.1" 422 Unprocessable Content
Status Code: 422
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "body",
        "idade"
      ],
      "msg": "Input should be greater than or equal to 18",
      "input": 15,
      "ctx": {
        "ge": 18
      }
    }
  ]
}


### 🛠️ Guia Passo a Passo: Commit e Push para o GitHub

Siga os passos abaixo no terminal do seu computador (ou no Git Bash/Terminal do VS Code) para versionar o seu projeto utilizando práticas recomendadas.

#### Passo 1: Baixe a estrutura criada no Colab
No painel lateral esquerdo do Colab (ícone de pasta), clique com o botão direito e faça o download de:
1. Pasta `app/` (com todos os arquivos `.py` dentro).
2. Pasta `models/` (com o arquivo `credit_model.joblib`).
3. Arquivo `requirements.txt`.

Coloque tudo dentro de uma pasta dedicada no seu computador (por exemplo: `projeto-analise-preditiva`).

---

#### Passo 2: Inicialize e configure o repositório local
Abra o terminal na pasta do seu projeto e execute:

```bash
# Inicializa o Git na pasta local
git init

# (Opcional) Configure seu nome e e-mail no Git se ainda não o fez
git config --global user.name "Seu Nome"
git config --global user.email "seu.email@exemplo.com"
```

---

#### Passo 3: Siga o fluxo de Branches (GitFlow)
Conforme exigido nas instruções do seu projeto, evite realizar commits diretos na `main`:

```bash
# 1. Cria e muda para a branch 'develop'
git checkout -b develop

# 2. Cria uma branch específica (feature branch) para a estrutura do projeto
git checkout -b feature/estrutura-modular
```

---

#### Passo 4: Adicione os arquivos e faça o Commit
Agora, vamos registrar suas alterações de forma organizada:

```bash
# Adiciona todas as pastas e arquivos para a área de preparação (Staging Area)
git add app/ models/ requirements.txt

# Confirma as alterações com uma mensagem objetiva e sucinta
git commit -m "implementa estrutura modular da api e persiste modelo de credito"
```

---

#### Passo 5: Merge para a Branch de Desenvolvimento
Após concluir as alterações na sua feature branch, junte o trabalho de volta na branch de integração:

```bash
# Retorna para a branch develop
git checkout develop

# Une as alterações de forma segura
git merge feature/estrutura-modular
```

---

#### Passo 6: Crie o Repositório no GitHub e envie os Arquivos (Push)
1. Vá ao seu perfil no [GitHub](https://github.com/) e clique em **New Repository**.
2. Dê um nome ao seu repositório (ex: `projeto-analise-preditiva`) e certifique-se de mantê-lo como **Público**.
3. Copie o link do repositório gerado (terminado em `.git`).
4. No seu terminal local, execute:

```bash
# Vincula o repositório do seu computador com o repositório criado no GitHub
git remote add origin https://github.com/seu-usuario/seu-repositorio.git

# Envia as branches criadas localmente para a nuvem
git push -u origin develop
```

*Dica de ouro:* Quando finalizar e validar que tudo funciona perfeitamente na branch `develop`, faça o merge final para a branch `main` e execute `git push origin main` para atualizar a produção!

# Documentação Final do Projeto: `README.md`

In [49]:
%%writefile README.md
# API de Análise Preditiva de Risco de Crédito (Score de Crédito)

Este repositório contém a implementação completa de um microsserviço assíncrono baseado em **FastAPI** para disponibilização de um modelo preditivo de Machine Learning treinado para a classificação e análise de risco/score de crédito.

Este projeto foi desenvolvido como requisito obrigatório para o **Projeto Avaliativo do Módulo 2 (Semana 13)** da disciplina de Desenvolvimento de IA para Análise Preditiva.

---

## 1. Problema de Negócio e Objetivo do Sistema

No setor financeiro, a velocidade e a precisão no processo de concessão de crédito são determinantes. O objetivo desta API é analisar dados cadastrais e financeiros básicos de um cliente em tempo real para predizer se ele apresenta alto risco de inadimplência (`inadimplente_predito = 1`) ou baixo risco (`inadimplente_predito = 0`).

O modelo preditivo analisa de forma combinada variáveis como **idade**, **renda anual**, **score de crédito** e **histórico prévio de inadimplência**.

---

## 2. Tecnologias Utilizadas

*   **Python 3.10+**
*   **FastAPI**: Framework web assíncrono moderno e de alto desempenho.
*   **Pydantic (v2)**: Validação e tipagem de dados estrita em tempo de execução.
*   **Scikit-Learn**: Treinamento do pipeline offline (`StandardScaler`, `SimpleImputer` e `RandomForestClassifier`).
*   **Joblib**: Serialização e desserialização eficiente do pipeline preditivo.
*   **Uvicorn**: Servidor web ASGI para execução de alta performance.
*   **Pandas & NumPy**: Pré-processamento e manipulação dos conjuntos de dados.

---

## 3. Estrutura Modular da API

A arquitetura do projeto segue um padrão desacoplado em camadas para facilitar a manutenção e escalabilidade:

```text
├── models/
│   └── credit_model.joblib      # Pipeline completo serializado (Fase 1)
├── app/
│   ├── __init__.py
│   ├── main.py                  # Ponto de entrada, configuração do ciclo de vida Lifespan (Fase 3)
│   ├── schemas.py               # Modelos Pydantic de entrada e saída (Fase 4)
│   ├── routers.py               # Endpoints da aplicação (/health e /predict) (Fase 5)
│   └── services.py              # Inicialização do modelo em memória RAM (Services)
├── requirements.txt             # Dependências determinísticas para reprodutibilidade
└── README.md                    # Esta documentação
```

---

## 4. Instruções de Instalação e Execução Local

Siga o passo a passo abaixo para rodar a aplicação localmente no seu computador:

### Passo 1: Clonar o repositório e acessar a pasta
```bash
git clone https://github.com/seu-usuario/seu-repositorio.git
cd seu-repositorio
```

### Passo 2: Criar e ativar um ambiente virtual (venv)
No Windows:
```bash
python -m venv venv
.\venv\Scripts\activate
```
No Linux/macOS:
```bash
python3 -m venv venv
source venv/bin/activate
```

### Passo 3: Instalar as dependências necessárias
```bash
pip install --upgrade pip
pip install -r requirements.txt
```

### Passo 4: Executar a API com o Uvicorn
```bash
uvicorn app.main:app --reload --host 127.0.0.1 --port 8000
```

Após o início do servidor, a API estará acessível em `http://127.0.0.1:8000`.

---

## 5. Documentação e Teste dos Endpoints

A documentação interativa oficial gerada automaticamente pelo FastAPI estará disponível em:
*   **Swagger UI**: `http://127.0.0.1:8000/docs`
*   **ReDoc**: `http://127.0.0.1:8000/redoc`

### 5.1 Endpoint de Saúde: `GET /health`
Rota de checagem que valida se a API está ativa e se o modelo do classificador reside corretamente na memória RAM.

**Exemplo de Resposta (Status `200 OK`):**
```json
{
  "status": "healthy",
  "model_loaded": true,
  "model_path": "models/credit_model.joblib"
}
```

### 5.2 Endpoint de Inferência Preditiva: `POST /predict`

#### Exemplo 1: Payload de Entrada Válido

**Requisição (`POST /predict`):**
```json
{
  "idade": 35,
  "renda_anual": 55000.0,
  "historico_inadimplencia": 0,
  "score_credito": 720
}
```

**Resposta Esperada (Status `200 OK`):**
```json
{
  "inadimplente_predito": 0,
  "probabilidade": 1.0
}
```

#### Exemplo 2: Payload de Entrada Inválido (Erro do Pydantic)
Idade informada como menor que `18` (regra mínima de negócio definida no Schema do Pydantic).

**Requisição (`POST /predict`):**
```json
{
  "idade": 12,
  "renda_anual": 25000.0,
  "historico_inadimplencia": 0,
  "score_credito": 600
}
```

**Resposta Esperada (Status `422 Unprocessable Entity`):**
```json
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "body",
        "idade"
      ],
      "msg": "Input should be greater than or equal to 18",
      "input": 12,
      "ctx": {
        "ge": 18
      }
    }
  ]
}
```

---

## 6. Links de Evidências (Entrega)

*   **Link do Repositório GitHub**: `https://github.com/seu-usuario/seu-repositorio` *(Substitua pelo seu link público)*
*   **Link da Vídeo-Apresentação (Google Drive)**: `https://drive.google.com/open?id=seu-link-de-compartilhamento` *(Substitua pelo seu link em modo leitor)*


Writing README.md
